Start by converting CSV files into dataframes.

In [4]:
import pandas as pd

# Convert CSV files into dataframes
orders = pd.read_csv('../dataset/orders.csv')
order_products_prior = pd.read_csv('../dataset/order_products__prior.csv')
order_products_train = pd.read_csv('../dataset/order_products__train.csv')
products = pd.read_csv('../dataset/products.csv')
aisles = pd.read_csv('../dataset/aisles.csv')
departments = pd.read_csv('../dataset/departments.csv')

Perform lightweight analysis on each individual dataframe.

In [5]:
def lightweight_eda(df):
    print(df.info())
    print("Number of missing values:")
    print(df.isna().sum())
    print("Number of duplicates:")
    print(df.duplicated().sum())


# Perform a lightweight EDA on each file
print("ORDERS")
print(orders['order_id'].value_counts())
lightweight_eda(orders)

print("PRODUCTS")
print(products['product_id'].value_counts())
lightweight_eda(products)

print("AISLES")
print(aisles['aisle_id'].value_counts())
lightweight_eda(aisles)

print("DEPARTMENTS")
print(departments['department_id'].value_counts())
lightweight_eda(departments)

print("ORDER_PRODUCTS: HISTORY")
lightweight_eda(order_products_prior)

print("ORDER_PRODUCTS: TEST")
lightweight_eda(order_products_train)

ORDERS
order_id
2539329    1
1591157    1
1354759    1
1971373    1
1558866    1
          ..
3266950    1
118963     1
9433       1
2938641    1
272231     1
Name: count, Length: 3421083, dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3421083 entries, 0 to 3421082
Data columns (total 7 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   order_id                int64  
 1   user_id                 int64  
 2   eval_set                object 
 3   order_number            int64  
 4   order_dow               int64  
 5   order_hour_of_day       int64  
 6   days_since_prior_order  float64
dtypes: float64(1), int64(5), object(1)
memory usage: 182.7+ MB
None
Number of missing values:
order_id                       0
user_id                        0
eval_set                       0
order_number                   0
order_dow                      0
order_hour_of_day              0
days_since_prior_order    206209
dtype: int64
Number of dupl

Merge all into a single flat dataframe.

In [7]:
products_full = (products
    .merge(aisles, on='aisle_id', how='left')
    .merge(departments, on='department_id', how='left')
)

prior_products = order_products_prior.merge(products_full, on='product_id', how='left')

full_orders_products = prior_products.merge(
    orders,
    on='order_id',
    how='left'
)

# Convert object dtypes for efficient manipulation
full_orders_products['product_name'] = full_orders_products['product_name'].astype('string')
full_orders_products['department'] = full_orders_products['department'].astype('category')
full_orders_products['aisle'] = full_orders_products['aisle'].astype('category')


print(full_orders_products.head())



   order_id  product_id  add_to_cart_order  reordered           product_name  \
0         2       33120                  1          1     Organic Egg Whites   
1         2       28985                  2          1  Michigan Organic Kale   
2         2        9327                  3          0          Garlic Powder   
3         2       45918                  4          1         Coconut Butter   
4         2       30035                  5          0      Natural Sweetener   

   aisle_id  department_id               aisle  department  user_id eval_set  \
0        86             16                eggs  dairy eggs   202279    prior   
1        83              4    fresh vegetables     produce   202279    prior   
2       104             13   spices seasonings      pantry   202279    prior   
3        19             13       oils vinegars      pantry   202279    prior   
4        17             13  baking ingredients      pantry   202279    prior   

   order_number  order_dow  order_hour

Check for duplicates.

In [8]:
# Checking if there are instances where a product appears in an order multiple times
duplicates_within_orders = (
    full_orders_products
    .duplicated(subset=['order_id', 'product_name'], keep=False)
)
print("Number of (order_id, product) duplicates:",
      duplicates_within_orders.sum())

Number of (order_id, product) duplicates: 0


Clean up unnecessary columns.

In [9]:
# Remove columns that are not needed after merging
columns_to_remove = ["add_to_cart_order","reordered", "order_dow", "order_hour_of_day", "days_since_prior_order", "eval_set"]

truncated_df = full_orders_products.drop(columns=columns_to_remove)

truncated_df.head()

truncated_df.to_csv('../data/cleaned/order-products-full.csv', index=False)

ORDER ANALYSIS

Order Size

In [12]:
import matplotlib.pyplot as plt
%run ../utils/categorize.py


order_info_df = orders[orders['eval_set'] == 'prior']
order_info_df = order_info_df[['order_id', 'user_id']]

# Calculate order size
order_sizes = truncated_df.groupby('order_id')['product_id'].count()
order_sizes = order_sizes.reset_index(name='order_size')
order_size_counts = order_sizes['order_size'].value_counts().sort_index()

# Statistics for order size
print(order_sizes['order_size'].describe())

# Distribution for order sizes
plt.bar(order_size_counts.index, order_size_counts.values, color='skyblue', edgecolor='black')
plt.title("Order Size Distribution")
plt.xlabel("Number of Products per Order")
plt.ylabel("Count")
plt.tight_layout()
plt.savefig(f"order_size_frequency.png") 
plt.close()

# Build dataframe for analysis
order_info_full_df = order_info_df.copy()
order_info_full_df = order_info_full_df.merge(order_sizes, on='order_id', how='left')

# Create a categorical column to represent order size
order_sizes['order_size_cat'] = categorize_by_distribution(order_sizes['order_size'])
order_info_df['order_size_cat'] = order_info_df['order_id'].map(
    order_sizes.set_index('order_id')['order_size_cat']
)

count    3.214874e+06
mean     1.008888e+01
std      7.525398e+00
min      1.000000e+00
25%      5.000000e+00
50%      8.000000e+00
75%      1.400000e+01
max      1.450000e+02
Name: order_size, dtype: float64


Orders per User

In [15]:
# Count number of orders per user
user_order_counts = order_info_df.groupby('user_id')['order_id'].nunique()
user_order_counts = user_order_counts.reset_index(name='user_order_count')

# Statistics for orders per user
print(user_order_counts['user_order_count'].describe())

# Distribution for orders per user
plt.hist(user_order_counts['user_order_count'], bins=50, color='skyblue', edgecolor='black')
plt.xlabel("Number of Orders per User")
plt.ylabel("Number of Users")
plt.title("Distribution of Orders per User")
plt.savefig(f"user_order_counts_hist.png") 
plt.close()

# Add orders per user to exisiting df
order_info_full_df = order_info_full_df.merge(user_order_counts, on='user_id', how='left')

# Create summary and save
order_summary = order_info_full_df.describe()
order_summary.to_csv('../data/summary/order-info-summary.csv', index=False)

# Create a categorical column to represent orders per user
user_order_counts['orders_per_user_cat'] = categorize_by_distribution(user_order_counts['user_order_count'])
order_info_df['orders_per_user_cat'] = order_info_df['user_id'].map(
    user_order_counts.set_index('user_id')['orders_per_user_cat']
)

# Save the dataframe for future use
order_info_df.to_csv('../data/cleaned/order-info.csv', index=False)

count    206209.000000
mean         15.590367
std          16.654774
min           3.000000
25%           5.000000
50%           9.000000
75%          19.000000
max          99.000000
Name: user_order_count, dtype: float64


PRODUCT ANALYSIS

Orders per Product

In [16]:
product_info_df = products.copy()
product_info_full_df = products.copy()

# Calculate number of orders a product has appeared in
total_orders = truncated_df['order_id'].nunique()
product_counts = truncated_df['product_id'].value_counts().reset_index()
product_counts.columns = ['product_id', 'count']

# Add column to show percentage of orders containing each product
product_counts['order_penetration_pct'] = (product_counts['count'] / total_orders) * 100

# Distribution for orders per product
plt.figure(figsize=(8, 5))
plt.hist(product_counts['count'], bins=50, color='skyblue', edgecolor='black')
plt.title("Distribution of Product Counts")
plt.xlabel("Number of Orders")
plt.ylabel("Number of Products")
plt.tight_layout()
plt.savefig(f"product_order_counts_hist.png") 
plt.close()

# Add columns for analysis
product_info_full_df = product_info_full_df.merge(product_counts, on='product_id', how='left')

# Create a categorical column to represent order counts per product
product_counts['orders_per_product_cat'] = categorize_by_distribution(product_counts['count'])
product_info_df['orders_per_product_cat'] = product_info_df['product_id'].map(
    product_counts.set_index('product_id')['orders_per_product_cat']
)

Orders per department.

In [17]:
import matplotlib.pyplot as plt

# Calculate number of orders a department has appeared in
total_orders = truncated_df['order_id'].nunique()
department_counts = (
    truncated_df[['order_id', 'department_id']]
    .drop_duplicates() 
    .groupby('department_id')['order_id']
    .nunique()
    .reset_index()
    .rename(columns={'order_id': 'count'})
)

# Add column to show percentage of orders containing each department
department_counts['dept_penetration_pct'] = (department_counts['count'] / total_orders) * 100

# Distribution for department counts
plt.figure(figsize=(8, 5))
plt.hist(department_counts['count'], bins=50, color='skyblue', edgecolor='black')
plt.title("Distribution of Department Counts")
plt.xlabel("Number of Orders")
plt.ylabel("Number of Departments")
plt.tight_layout()
plt.savefig(f"dept_order_counts_hist.png") 
plt.close()

# Add columns for analysis
product_info_full_df = product_info_full_df.merge(department_counts, on='department_id', how='left')

# Create a categorical column to represent order counts per department
department_counts['orders_per_dept_cat'] = categorize_by_distribution(department_counts['count'])
product_info_df['orders_per_dept_cat'] = product_info_df['department_id'].map(
    department_counts.set_index('department_id')['orders_per_dept_cat']
)

Orders per Aisle.

In [18]:
# Calculate number of orders an aisle has appeared in
total_orders = truncated_df['order_id'].nunique()
aisle_counts = (
    truncated_df[['order_id', 'aisle_id']]
    .drop_duplicates()  
    .groupby('aisle_id')['order_id']
    .nunique()
    .reset_index()
    .rename(columns={'order_id': 'count'})
)

# Add column to show percentage of orders containing each aisle
aisle_counts['aisle_penetration_pct'] = (aisle_counts['count'] / total_orders) * 100

# Distribution for aisle counts
plt.figure(figsize=(8, 5))
plt.hist(department_counts['count'], bins=50, color='skyblue', edgecolor='black')
plt.title("Distribution of Aisle Counts")
plt.xlabel("Number of Orders")
plt.ylabel("Number of Aisles")
plt.tight_layout()
plt.savefig(f"aisle_order_counts_hist.png") 
plt.close()

# Add columns for analysis
product_info_full_df = product_info_full_df.merge(aisle_counts, on='aisle_id', how='left')

# Create summary and save
product_summary = product_info_full_df.describe()
product_summary.to_csv('../data/summary/product-info-summary.csv', index=False)

# Create a categorical column to represent order counts per aisle
aisle_counts['orders_per_aisle_cat'] = categorize_by_distribution(aisle_counts['count'])
product_info_df['orders_per_aisle_cat'] = product_info_df['aisle_id'].map(
    aisle_counts.set_index('aisle_id')['orders_per_aisle_cat']
)

# Save the dataframe for future use
product_info_df.to_csv('../data/cleaned/product-info.csv', index=False)

EDA Report

In [19]:
from ydata_profiling import ProfileReport


# Build a sample of users to keep order integrity and product diversity
sample_frac = 0.05
sampled_users = orders['user_id'].drop_duplicates().sample(frac=sample_frac, random_state=42)
df_sample = full_orders_products[full_orders_products['user_id'].isin(sampled_users)]

profile = ProfileReport(df_sample, explorative=True)
profile.to_file("eda_report_orders.html")

c:\Users\jenle\AppData\Local\Programs\Python\Python312\Lib\site-packages\ydata_profiling\utils\dataframe.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df.rename(columns={"index": "df_index"}, inplace=True)


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 15/15 [00:09<00:00,  1.67it/s]


Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

ADDITIONAL DATA PREPARATION

Build and save product pair probabilities.

In [20]:
%run ../utils/pairwise.py
%run ../utils/sampling.py

sampled_products = stratified_product_sampling_fixed_size(
    product_counts,
    target_sample_size=5000,
    min_orders=50
)

sampled_products_df = pd.DataFrame(sampled_products, columns=['product_id'])
sampled_products_df.to_csv('../data/sample/sampled-products.csv', index=False)

order_product_df = order_products_prior[['order_id', 'product_id']]

product_pair_df = compute_pairwise_probabilities_sample(order_product_df, 
    sampled_products,
    output_csv="../data/sample/sample-pairwise.csv",
    batch_size=1000
)

Sampled 5000 products from 26686 eligible products.


100%|██████████| 5/5 [03:13<00:00, 38.73s/it]

Completed computation. Saved to ../data/sample/sample-pairwise.csv


Build test set.

In [21]:
test_df = order_products_train.merge(orders, on='order_id', how='left')

order_products_users_df = test_df[['order_id', 'product_id', 'user_id']]
order_products_users_df.to_csv('../data/cleaned/order-products-users-test.csv', index=False)

order_products_df = test_df[['order_id', 'product_id']]
order_products_df.to_csv('../data/cleaned/order-products-test.csv', index=False)

Build a set of "discontinued" products (products not found in users' last order).

In [22]:
# Get unique products from each df
prior_products = set(order_products_prior["product_id"].unique())
train_products = set(order_products_train["product_id"].unique())

# Find products in prior but not in train 
discontinued_products = prior_products - train_products

print(f"Number of products in prior only: {len(discontinued_products)}")

discontinued_df = pd.DataFrame({"product_id": list(discontinued_products)})

#  Merge to get order counts for discontinued products
discontinued_with_counts = discontinued_df.merge(product_counts, on="product_id", how="left")

# Get the top 25% of products
top_discontinued_products = discontinued_with_counts[discontinued_with_counts["count"] > 19]

pd.DataFrame({"product_id": list(top_discontinued_products)}).to_csv(
    "../data/cleaned/top-discontinued-products.csv", index=False
)

Number of products in prior only: 10562


Build pairwise probabilities for discontinued products.

In [24]:
order_products_prior = pd.read_csv('../dataset/order_products__prior.csv')

order_product_df = order_products_prior[['order_id', 'product_id']]

product_pair_df = compute_pairwise_probabilities_sample(order_product_df, 
    list(top_discontinued_products["product_id"]),
    output_csv="../data/discontinued/discontinued-pairwise.csv",
    batch_size=1000
)

100%|██████████| 3/3 [00:16<00:00,  5.44s/it]

Completed computation. Saved to ../data/discontinued/discontinued-pairwise.csv
